# vllm

> Bring up a vLLM OpenAI-compatible endpoint on a SLURM compute node and forward it to localhost. Reuses the four primitives from `slurm_ops.core` (`start_or_connect`, `job_stat`, `get_port_forwarding_command`, `update_ssh_node_config`); adds an `apptainer exec --nv vllm.sif` wrapper so the GPU-side runtime is reproducible across clusters.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp vllm

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_ne, test_fail

In [ ]:
#| export
import json
import shlex
import subprocess
import time
from pathlib import Path

from .core import (
    start_or_connect,
    job_stat,
    get_port_forwarding_command,
    update_ssh_node_config,
)

In [ ]:
#| export
DEFAULT_SLURM_ARGS = (
    "--account=amath --partition=gpu-l40s --gres=gpu:1 "
    "--cpus-per-task=8 --mem=48G --time=04:00:00"
)
DEFAULT_MODEL = "Qwen/Qwen3-8B"

### `sif_exists` — is the apptainer image on the cluster?

In [ ]:
#| export
def sif_exists(host, sif_path="/mmfs1/gscratch/scrubbed/$USER/vllm.sif"):
    "True iff the apptainer image is on `host`."
    r = subprocess.run(
        ["ssh", host, f"[ -f {sif_path} ] && echo yes || echo no"],
        capture_output=True, text=True,
    )
    return r.stdout.strip() == "yes"

### `build_sif` — submit the SIF build sbatch and block until done

In [ ]:
#| export
def build_sif(host, remote_vllm_dir="projects/gcd/slurm-ops/vllm", silent=False):
    """Submit ~/<remote_vllm_dir>/build-sif.job on `host` and block until done.

    Returns the slurm exit code (0 = sif built). The first build pulls a ~10GB
    docker image and takes ~25-35 min; subsequent calls are no-ops because of
    sif_exists().
    """
    if not silent:
        print(f"[vllm] submitting build-sif.job on {host} (~25-35 min first time)...")
    sub = subprocess.run(
        ["ssh", host, f"cd ~/{remote_vllm_dir} && sbatch --parsable build-sif.job"],
        capture_output=True, text=True, check=True,
    )
    jobid = sub.stdout.strip()
    if not jobid:
        raise RuntimeError(f"sbatch returned no job id; stderr:\n{sub.stderr}")
    if not silent:
        print(f"[vllm] SIF build job: {jobid}; tailing slurm-{jobid}.out (Ctrl-C only stops the tail)")
    tail_script = (
        f"log=~/{remote_vllm_dir}/slurm-{jobid}.out; "
        f"for _ in $(seq 1 600); do [ -f \"$log\" ] && break; sleep 2; done; "
        f"( tail -n +1 -f \"$log\" ) & tp=$!; "
        f"while squeue -h -j {jobid} 2>/dev/null | grep -q .; do sleep 5; done; "
        f"sleep 2; kill $tp 2>/dev/null || true; wait $tp 2>/dev/null || true; "
        f"ec=$(sacct -j {jobid}.batch -o ExitCode -P -n 2>/dev/null | head -1 | cut -d: -f1); "
        f"exit ${{ec:-0}}"
    )
    r = subprocess.run(["ssh", host, tail_script])
    return r.returncode

### `wait_for_discovery` — block until serve.sh publishes the endpoint

In [ ]:
#| export
def wait_for_discovery(job_name, host, timeout_s=1800, poll_s=5, silent=False):
    """Block until ~/.vllm-discovery/<job_name>.json exists on `host`.

    Returns the parsed JSON dict ({\"node\": ..., \"port\": ..., \"model\": ...}).
    Bails early if the SLURM job leaves the queue before the file appears.
    """
    disc_path = f"$HOME/.vllm-discovery/{job_name}.json"
    deadline = time.time() + timeout_s
    if not silent:
        print(f"[vllm] waiting for {disc_path} on {host} (up to {timeout_s//60} min)...")
    while time.time() < deadline:
        r = subprocess.run(
            ["ssh", host, f"cat {disc_path} 2>/dev/null || true"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            return json.loads(r.stdout)
        if job_stat(job_name, host, silent=True) is None:
            log = subprocess.run(
                ["ssh", host, f"tail -40 $HOME/.vllm-discovery/{job_name}.log 2>/dev/null"],
                capture_output=True, text=True,
            ).stdout
            raise RuntimeError(
                f"job '{job_name}' is no longer in the queue and no discovery file appeared.\n"
                f"Recent serve log:\n{log}"
            )
        time.sleep(poll_s)
    raise TimeoutError(f"timed out waiting for {disc_path}")

### `vllm_up` — full bring-up

Wires the four `slurm_ops.core` primitives together. The only material change vs upstream's pattern is that we *execute* the commands those functions return, and the salloc body ends in `srun --pty ~/<remote_vllm_dir>/serve.sh` (which runs `apptainer exec --nv vllm.sif python3 -m vllm.entrypoints.openai.api_server`) rather than a bare shell.

In [ ]:
#| export
def vllm_up(
    job_name,
    host,
    model=DEFAULT_MODEL,
    slurm_args=DEFAULT_SLURM_ARGS,
    local_port=8000,
    remote_vllm_dir="projects/gcd/slurm-ops/vllm",
    update_node_config=True,
):
    """Bring up a vLLM OpenAI-compatible endpoint on `host` and forward it to localhost.

    Wires together the upstream slurm-ops primitives:
      - `start_or_connect` (re-uses or starts the tmux+salloc session)
      - `job_stat`          (resolves the compute node after allocation)
      - `get_port_forwarding_command` (composes the ssh -L command)
      - `update_ssh_node_config`      (rewrites ~/.ssh/<cluster>-node-config so
                                       VSCode can attach to the compute node)
    """
    if not sif_exists(host):
        rc = build_sif(host, remote_vllm_dir=remote_vllm_dir)
        if rc != 0 or not sif_exists(host):
            raise RuntimeError(f"SIF build failed (exit {rc}); see slurm-*.out in ~/{remote_vllm_dir}/")

    serve_cmd = f"srun --pty ~/{remote_vllm_dir}/serve.sh"
    full_slurm_args = f"{slurm_args} {serve_cmd}"

    ssh_cmd = start_or_connect(job_name, host, slurm_args=full_slurm_args, return_cmd=True)
    if model != DEFAULT_MODEL:
        ssh_cmd = ssh_cmd.replace(
            "tmux new-session -A -s",
            f"MODEL={shlex.quote(model)} tmux new-session -A -s",
            1,
        )

    ssh_cmd_detached = ssh_cmd.replace("tmux new-session -A", "tmux new-session -A -d", 1)
    print(f"[vllm] launching: {ssh_cmd_detached}")
    subprocess.run(ssh_cmd_detached, shell=True, check=True)

    print(f"[vllm] waiting for salloc to allocate a GPU node...")
    for _ in range(360):
        info = job_stat(job_name, host, silent=True)
        if info is not None:
            node, jobid = info
            print(f"[vllm] allocated: job {jobid} on {node}")
            break
        time.sleep(5)
    else:
        raise TimeoutError(f"salloc never reached RUNNING state for '{job_name}'")

    disc = wait_for_discovery(job_name, host)
    node = disc["node"]
    remote_port = disc["port"]

    _ = get_port_forwarding_command(local_port, remote_port, host, node)
    of_cmd = f"ssh -O forward -L {local_port}:{node}.hyak.local:{remote_port} {host}"
    print(f"[vllm] opening forward: {of_cmd}")
    subprocess.run(of_cmd, shell=True, check=True)

    if update_node_config:
        try:
            update_ssh_node_config(job_name, host)
        except (RuntimeError, FileNotFoundError) as e:
            print(f"[vllm] (skipped update_ssh_node_config: {e})")

    base_url = f"http://localhost:{local_port}/v1"
    print()
    print("=" * 60)
    print(f"  vLLM ready: {disc['model']} on {node}:{remote_port}")
    print(f"  base_url = {base_url}")
    print(f"  export OPENAI_BASE_URL={base_url!r}")
    print(f"  export OPENAI_API_KEY='dummy'")
    print(f"  export VLLM_MODEL={disc['served_name']!r}")
    print("=" * 60)
    return {
        "base_url": base_url,
        "model": disc["model"],
        "served_name": disc["served_name"],
        "node": node,
        "local_port": local_port,
        "remote_port": remote_port,
        "job_name": job_name,
        "host": host,
    }

### `vllm_down` — tear down

In [ ]:
#| export
def vllm_down(job_name, host, local_port=8000, remote_port=None, node=None):
    """Tear down a vllm_up-style session.

    Cancels the slurm job, kills the tmux session, and cancels the local forward.
    Idempotent.
    """
    info = job_stat(job_name, host, silent=True)
    if (node is None or remote_port is None):
        r = subprocess.run(
            ["ssh", host, f"cat $HOME/.vllm-discovery/{job_name}.json 2>/dev/null || true"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            d = json.loads(r.stdout)
            node = node or d.get("node")
            remote_port = remote_port or d.get("port")

    if node and remote_port:
        cancel = f"ssh -O cancel -L {local_port}:{node}.hyak.local:{remote_port} {host}"
        print(f"[vllm] {cancel}")
        subprocess.run(cancel, shell=True)
    else:
        print("[vllm] (no node/remote_port known; skipping local forward cancel)")

    if info is not None:
        _, jobid = info
        print(f"[vllm] scancel {jobid}")
        subprocess.run(["ssh", host, f"scancel {jobid}"])
    else:
        print(f"[vllm] no '{job_name}' job in queue")

    subprocess.run(["ssh", host, f"tmux kill-session -t {job_name} 2>/dev/null || true"])
    print("[vllm] done.")

## Usage

```python
from slurm_ops.vllm import vllm_up, vllm_down
info = vllm_up("gcd", "klone-login")        # blocks until ready
# ... call info['base_url'] from your code ...
vllm_down("gcd", "klone-login")
```

For CLI users, two thin shell stubs in `vllm/bin/` invoke these from the command line.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()